## Data loading

In [ ]:
# ── Set your file path here ───────────────────────────────
DATASET_PATH = "BanglishRev.json.gz"   # update to your local path

import os
file_size_mb = os.path.getsize(DATASET_PATH) / (1024 ** 2)
print(f"✅ File found: {DATASET_PATH}")
print(f"   Size: {file_size_mb:.1f} MB")

## Install Dependancies

In [ ]:
!pip install implicit scikit-learn pandas numpy scipy tabulate --quiet
print("✅ All packages ready.")

## Imports

In [ ]:
import json, math, warnings
import numpy  as np
import pandas as pd
import scipy.sparse as sp

from scipy.sparse.linalg          import svds
from sklearn.metrics.pairwise     import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from tabulate                     import tabulate
from google.colab                 import files
from sklearn.utils.extmath import randomized_svd as sklearn_rsvd

warnings.filterwarnings("ignore")
print("✅ Imports done.")

## Data Loading and Filtering

In [ ]:
import gzip
import orjson
import os
import time
import multiprocessing as mp
import numpy as np
import pandas as pd

# ── Config ────────────────────────────────────────────────
CHUNK_SIZE = 5_000
N_WORKERS  = max(1, mp.cpu_count() - 1)

print(f"📦  Dataset  : {DATASET_PATH}")
print(f"🧠  Workers  : {N_WORKERS}  |  Chunk size : {CHUNK_SIZE:,}")


# ── Step 1: Load compressed JSON ──────────────────────────
print("\n⏳ Loading JSON …", end=" ")
t0 = time.time()

with gzip.open(DATASET_PATH, "rb") as f:
    raw_data = orjson.loads(f.read())

print(f"done in {time.time()-t0:.1f}s  ({len(raw_data):,} products)")


# ── Step 2: Worker function ───────────────────────────────
def flatten_chunk(product_chunk: list) -> list:
    """
    Flattens a chunk of product dicts into flat row dicts.
    Only extracts the 5 columns needed by the active models.
    """
    rows = []
    for product in product_chunk:
        pid      = str(product.get("Product ID", ""))
        category = product.get("Category", "Unknown")
        reviews  = product.get("Reviews") or []

        if not pid:
            continue

        rows.extend([
            {
                "user_id":    str(r["Buyer ID"]),
                "product_id": pid,
                "rating":     r.get("Current Rating"),
                "category":   category,
                "timestamp":  r.get("Review Date"),
            }
            for r in reviews
            if r.get("Buyer ID") is not None
            and r.get("Current Rating") is not None
        ])

    return rows


# ── Step 3: Parallel flatten ──────────────────────────────
def chunked(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i: i + size]

chunks = list(chunked(raw_data, CHUNK_SIZE))
print(f"\n⏳ Flattening {len(chunks)} chunks across {N_WORKERS} workers …")

t1 = time.time()
with mp.Pool(processes=N_WORKERS) as pool:
    nested = pool.map(flatten_chunk, chunks)

all_rows = [row for chunk_result in nested for row in chunk_result]
print(f"✅ Flattening done in {time.time()-t1:.1f}s  ({len(all_rows):,} rows)")


# ── Step 4: Build DataFrame — only the 5 needed columns ──
print("\n⏳ Building DataFrame …", end=" ")
t2 = time.time()

df_raw = pd.DataFrame(all_rows, columns=[
    "user_id", "product_id", "rating", "category", "timestamp"
])

# Free raw data immediately
del all_rows, nested, raw_data
import gc; gc.collect()

print(f"done in {time.time()-t2:.1f}s")
print(f"\n✅ df_raw shape   : {df_raw.shape}")
print(f"   Unique users   : {df_raw['user_id'].nunique():,}")
print(f"   Unique items   : {df_raw['product_id'].nunique():,}")
print(f"   Columns kept   : {list(df_raw.columns)}")
df_raw.head(3)

## RAM sanity check

In [ ]:
# ================================================================================
# This cell audits memory and optionally drops heavy columns before preprocessing.
# ================================================================================

def df_memory_report(df: pd.DataFrame):
    """Prints per-column and total RAM usage."""
    mem = df.memory_usage(deep=True)
    total_mb = mem.sum() / 1024**2
    col_report = (
        mem[1:]                         # drop 'Index' entry
        .sort_values(ascending=False)
        .apply(lambda x: f"{x/1024**2:.1f} MB")
    )
    print("── DataFrame Memory Usage ───────────────")
    for col, usage in col_report.items():
        print(f"  {col:<20} {usage}")
    print(f"  {'TOTAL':<20} {total_mb:.1f} MB")
    print("─────────────────────────────────────────")

df_memory_report(df_raw)

# ── Optional: shrink string columns with categoricals ─────
# category, parent_cat, root_cat have very low cardinality
# → converting to Categorical saves ~10x memory on those columns
for col in ["category", "parent_cat", "root_cat"]:
    if col in df_raw.columns:
        df_raw[col] = df_raw[col].astype("category")

print("\nAfter categorical conversion:")
df_memory_report(df_raw)

## Pre-processing

In [ ]:
MIN_USER_INTERACTIONS = 10 # options tested: 5, 10, 15, 15, 20
MIN_ITEM_INTERACTIONS = 3  # options tested: 3,  3,  3,  5,  5

def preprocess(df: pd.DataFrame, min_user: int = 10, min_item: int = 3) -> pd.DataFrame:
    df = df.copy()

    # ── Cast rating to numeric ────────────────────────────
    if df["rating"].dtype == object:
        df["rating"] = df["rating"].astype(str).str.strip()
    df["rating"] = pd.to_numeric(df["rating"], errors="coerce")

    # ── Parse timestamps ──────────────────────────────────
    df["timestamp"] = pd.to_datetime(
        df["timestamp"], format="%Y-%m-%d", errors="coerce"
    )

    # ── Drop rows with missing rating OR timestamp ────────
    before = len(df)
    df.dropna(subset=["user_id", "product_id", "rating", "timestamp"], inplace=True)
    print(f"  Dropped {before - len(df):,} rows with missing rating/timestamp")

    # ── Clip to valid range and cast ──────────────────────
    df["rating"] = df["rating"].clip(1, 5).astype(np.float32)

    # ── Deduplicate: same user reviewing same product twice
    #    Keep highest rating (most positive signal) ────────
    before = len(df)
    df = (
        df.sort_values("rating", ascending=False)
          .drop_duplicates(subset=["user_id", "product_id"], keep="first")
    )
    print(f"  Dropped {before - len(df):,} duplicate (user, product) pairs")

    # ── Low-memory: category as Categorical dtype ─────────
    df["category"] = df["category"].astype("category")

    # ── Convergent iterative k-core filtering ─────────────
    # Runs until no more rows are removed (guaranteed every
    # user >= min_user and every item >= min_item).
    previous_size = -1
    iteration = 0
    while previous_size != len(df):
        previous_size = len(df)
        iteration += 1

        u_counts = df["user_id"].value_counts()
        df = df[df["user_id"].isin(u_counts[u_counts >= min_user].index)]

        i_counts = df["product_id"].value_counts()
        df = df[df["product_id"].isin(i_counts[i_counts >= min_item].index)]

        removed = previous_size - len(df)
        print(f"  Sparsity pass {iteration}: removed {removed:,} rows")

    df.reset_index(drop=True, inplace=True)

    # ── Sanity check ──────────────────────────────────────
    assert np.isfinite(df["rating"].values).all(), \
        "\u274c Non-finite ratings detected \u2014 check input data"
    assert not df.duplicated(subset=["user_id", "product_id"]).any(), \
        "\u274c Duplicate (user, product) pairs remain"

    return df


df = preprocess(df_raw, min_user=MIN_USER_INTERACTIONS, min_item=MIN_ITEM_INTERACTIONS)

print(f"\n\u2705 After preprocessing:")
print(f"   Rows         : {len(df):,}")
print(f"   Unique users : {df['user_id'].nunique():,}")
print(f"   Unique items : {df['product_id'].nunique():,}")
print(f"   Rating range : [{df['rating'].min():.0f}, {df['rating'].max():.0f}]")
print(f"   Memory usage : {df.memory_usage(deep=True).sum()/1024**2:.1f} MB")
df.head(3)


## 3-way temporal train / validation / test split

In [ ]:
def temporal_split_three_way(df: pd.DataFrame):
    """
    Returns train_df, val_df, test_df.
    All three are guaranteed to have no overlap by user-item pair.
    """
    df = df.copy()
    df.sort_values(["user_id", "timestamp"], inplace=True, na_position="last")

    # ── Rank interactions per user from the end ────────────
    # rank 1 = most recent, rank 2 = second most recent, etc.
    df["recency_rank"] = (
        df.groupby("user_id")
          .cumcount(ascending=False) + 1
    )

    test_mask  = df["recency_rank"] == 1
    val_mask   = df["recency_rank"] == 2
    train_mask = df["recency_rank"] >= 3

    train_df = df[train_mask].drop(columns="recency_rank").reset_index(drop=True)
    val_df   = df[val_mask  ].drop(columns="recency_rank").reset_index(drop=True)
    test_df  = df[test_mask ].drop(columns="recency_rank").reset_index(drop=True)

    return train_df, val_df, test_df


train_df, val_df, test_df = temporal_split_three_way(df)

# ── Coverage report ────────────────────────────────────────
total_users = df["user_id"].nunique()

# Users with >= 3 interactions appear in all three splits
# Users with exactly 2 appear in train + val only
# Users with exactly 1 appear in train only

train_only_users = set(train_df["user_id"]) - set(val_df["user_id"])
val_and_test     = set(val_df["user_id"]) & set(test_df["user_id"])

print(f"✅ Split complete")
print(f"{'─'*50}")
print(f"  {'Split':<10} {'Rows':>10}  {'Users':>10}")
print(f"{'─'*50}")
print(f"  {'Train':<10} {len(train_df):>10,}  {train_df['user_id'].nunique():>10,}")
print(f"  {'Val':<10} {len(val_df):>10,}  {val_df['user_id'].nunique():>10,}")
print(f"  {'Test':<10} {len(test_df):>10,}  {test_df['user_id'].nunique():>10,}")
print(f"{'─'*50}")
print(f"  Total unique users in df : {total_users:,}")
print(f"  Users in val AND test    : {len(val_and_test):,}  (full evaluation pipeline)")
print(f"  Users in train only      : {len(train_only_users):,}  (< 3 interactions)")

# ── Leakage check ─────────────────────────────────────────
# No (user, product) pair should appear in more than one split
train_pairs = set(zip(train_df["user_id"], train_df["product_id"]))
val_pairs   = set(zip(val_df["user_id"],   val_df["product_id"]))
test_pairs  = set(zip(test_df["user_id"],  test_df["product_id"]))

tv_overlap  = train_pairs & val_pairs
tt_overlap  = train_pairs & test_pairs
vt_overlap  = val_pairs   & test_pairs

print(f"\n── Leakage Check ─────────────────────────────────")
print(f"  Train ∩ Val  : {len(tv_overlap):,}  {'✅' if len(tv_overlap)  == 0 else '❌ LEAK'}")
print(f"  Train ∩ Test : {len(tt_overlap):,}  {'✅' if len(tt_overlap)  == 0 else '❌ LEAK'}")
print(f"  Val   ∩ Test : {len(vt_overlap):,}  {'✅' if len(vt_overlap)  == 0 else '❌ LEAK'}")

## CSR sparse matrix + bidirectional index maps

In [ ]:
def build_sparse_matrix(train_df: pd.DataFrame):
    """
    Returns
    ───────
    matrix   : scipy.sparse.csr_matrix  shape (n_users × n_items)
                 values = interaction score
    user2idx : {user_id_str  → row_int}
    item2idx : {product_id_str → col_int}
    idx2user : {row_int → user_id_str}
    idx2item : {col_int → product_id_str}
    """
    users = train_df["user_id"].unique()
    items = train_df["product_id"].unique()

    user2idx = {u: i for i, u in enumerate(users)}
    item2idx = {it: i for i, it in enumerate(items)}
    idx2user = {i: u  for u, i in user2idx.items()}
    idx2item = {i: it for it, i in item2idx.items()}

    row  = train_df["user_id"].map(user2idx).values
    col  = train_df["product_id"].map(item2idx).values
    data = train_df["rating"].values.astype(np.float32)

    matrix = sp.csr_matrix(
        (data, (row, col)),
        shape=(len(users), len(items)),
        dtype=np.float32,
    )
    return matrix, user2idx, item2idx, idx2user, idx2item


matrix, user2idx, item2idx, idx2user, idx2item = build_sparse_matrix(train_df)

density = matrix.nnz / (matrix.shape[0] * matrix.shape[1])
print(f"✅ Matrix shape : {matrix.shape[0]:,} users × {matrix.shape[1]:,} items")
print(f"   Non-zeros    : {matrix.nnz:,}")
print(f"   Density      : {density:.5%}")

## Matrix properties

In [ ]:
print("── Matrix Health Check ──────────────────────────────")
n_users, n_items = matrix.shape
print(f"  Shape          : {n_users:,} users × {n_items:,} items")
print(f"  Non-zeros      : {matrix.nnz:,}")
print(f"  Density        : {matrix.nnz / (n_users * n_items):.5%}")

# ARPACK hard constraint: k must be < min(n_rows, n_cols) - 1
max_safe_k = min(n_users, n_items) - 2
print(f"\n  min(n_users, n_items) : {min(n_users, n_items):,}")
print(f"  Max safe k for svds   : {max_safe_k:,}")
print(f"  Requested k=50        : {'✅ OK' if max_safe_k >= 50 else '❌ TOO LARGE — root cause confirmed'}")

# Check for empty rows/cols (ARPACK instability source)
row_sums = np.array(matrix.sum(axis=1)).flatten()
col_sums = np.array(matrix.sum(axis=0)).flatten()
print(f"\n  Zero-sum rows (users) : {(row_sums == 0).sum():,}")
print(f"  Zero-sum cols (items) : {(col_sums == 0).sum():,}")
print(f"  Min row sum           : {row_sums.min():.4f}")
print(f"  Min col sum           : {col_sums.min():.4f}")
print("─────────────────────────────────────────────────────")

## Recommender Sytem Models

In [ ]:
# ──────────────────────────────────────────────────────────
# Helper — L2-normalise rows of a CSR matrix (in-place safe)
# ──────────────────────────────────────────────────────────
def l2_normalize_rows(csr: sp.csr_matrix) -> sp.csr_matrix:
    norms = np.array(csr.power(2).sum(axis=1)).flatten() ** 0.5
    norms[norms == 0] = 1e-9
    return sp.diags(1.0 / norms) @ csr


# ══════════════════════════════════════════════════════════
# 1. GLOBAL POPULARITY RECOMMENDER
# ══════════════════════════════════════════════════════════
class GlobalPopularityRecommender:
    """
    Non-personalised baseline.
    Items ranked by total aggregated interaction score across all users.
    """

    def fit(self, train_df, matrix=None, user2idx=None,
            item2idx=None, idx2item=None):
        self.item2idx = item2idx

        self.user_seen = (
            train_df.groupby("user_id")["product_id"]
            .apply(set).to_dict()
        )
        self.popular = (
            train_df.groupby("product_id")["rating"]
            .sum()
            .sort_values(ascending=False)
            .index.tolist()
        )
        return self

    def recommend(self, user_id: str, k: int = 10) -> list:
        seen = self.user_seen.get(user_id, set())
        return [p for p in self.popular if p not in seen][:k]

    def score_all_items(self, user_id: str) -> np.ndarray:
        scores = np.zeros(len(self.item2idx), dtype=np.float32)
        n = len(self.popular)
        for rank, item in enumerate(self.popular):
            idx = self.item2idx.get(item, -1)
            if idx >= 0:
                scores[idx] = n - rank
        return scores


# ══════════════════════════════════════════════════════════
# 2. CATEGORY-BASED POPULARITY RECOMMENDER
# ══════════════════════════════════════════════════════════
class CategoryPopularityRecommender:
    """
    Personalised only by the user's dominant category preference.
    Recommends top items within that category; fills remainder from
    the global popular list.
    """

    def fit(self, train_df, matrix=None, user2idx=None,
            item2idx=None, idx2item=None):
        self.item2idx = item2idx

        self.user_seen = (
            train_df.groupby("user_id")["product_id"]
            .apply(set).to_dict()
        )

        # Per-user dominant category (mode)
        self.user_cat = (
            train_df.groupby("user_id")["category"]
            .agg(lambda x: x.mode().iat[0])
            .to_dict()
        )

        # Category → sorted item list
        self.cat_items = (
            train_df.groupby(["category", "product_id"])["rating"]
            .sum().reset_index()
            .sort_values("rating", ascending=False)
            .groupby("category")["product_id"]
            .apply(list).to_dict()
        )

        # Global fallback
        self.global_popular = (
            train_df.groupby("product_id")["rating"]
            .sum().sort_values(ascending=False).index.tolist()
        )
        return self

    def recommend(self, user_id: str, k: int = 10) -> list:
        seen = self.user_seen.get(user_id, set())
        cat  = self.user_cat.get(user_id, "")
        recs = [p for p in self.cat_items.get(cat, []) if p not in seen]

        if len(recs) < k:
            extra = [p for p in self.global_popular if p not in seen and p not in recs]
            recs  = (recs + extra)[:k]

        return recs[:k]

    def score_all_items(self, user_id: str) -> np.ndarray:
        scores = np.zeros(len(self.item2idx), dtype=np.float32)
        n = len(self.global_popular)

        # Base score from global popularity rank
        for rank, item in enumerate(self.global_popular):
            idx = self.item2idx.get(item, -1)
            if idx >= 0:
                scores[idx] = n - rank

        # Category boost — always outranks any non-category item
        cat = self.user_cat.get(user_id, "")
        for rank, item in enumerate(self.cat_items.get(cat, [])):
            idx = self.item2idx.get(item, -1)
            if idx >= 0:
                scores[idx] += n

        return scores

# ══════════════════════════════════════════════════════════
# 3. USER-BASED COLLABORATIVE FILTERING
# ══════════════════════════════════════════════════════════
class UserBasedCF:
    """
    Finds the top-N most similar users (cosine similarity on the
    normalised score matrix) and scores unseen items by a
    similarity-weighted sum of their interactions.
    """

    def __init__(self, n_neighbors: int = 50):
        self.n_neighbors = n_neighbors

    def fit(self, train_df, matrix, user2idx, item2idx, idx2item):
        self.matrix    = matrix
        self.user2idx  = user2idx
        self.item2idx  = item2idx
        self.idx2item  = idx2item
        self.user_seen = (
            train_df.groupby("user_id")["product_id"]
            .apply(set).to_dict()
        )
        # Normalised matrix for dot-product = cosine similarity
        self.norm_matrix = l2_normalize_rows(matrix)
        return self

    def recommend(self, user_id: str, k: int = 10) -> list:
        if user_id not in self.user2idx:
            return []

        u_idx    = self.user2idx[user_id]
        user_vec = self.norm_matrix[u_idx]                   # (1 × items)

        # Cosine similarity to all users
        sims          = np.array(self.norm_matrix @ user_vec.T).flatten()
        sims[u_idx]   = -1                                   # exclude self

        neighbors     = np.argpartition(sims, -self.n_neighbors)[-self.n_neighbors:]
        neighbors     = neighbors[np.argsort(sims[neighbors])[::-1]]
        neighbor_sims = sims[neighbors]                      # (N,)

        # Weighted sum of neighbour interactions → item scores
        neigh_matrix = self.matrix[neighbors]                # (N × items)
        item_scores  = np.array(
            neigh_matrix.T @ neighbor_sims[:, np.newaxis]
        ).flatten()                                          # (items,)

        # Mask seen items
        seen = self.user_seen.get(user_id, set())
        for p in seen:
            if p in self.item2idx:
                item_scores[self.item2idx[p]] = -np.inf

        top_k = np.argsort(item_scores)[-k:][::-1]
        return [self.idx2item[i] for i in top_k if item_scores[i] > -np.inf]


# ══════════════════════════════════════════════════════════
# 4. ITEM-BASED COLLABORATIVE FILTERING
# ══════════════════════════════════════════════════════════
class ItemBasedCF:
    """
    Scores unseen items by the cosine-similarity-weighted sum of
    the target user's own interaction scores with related items.
    """

    def fit(self, train_df, matrix, user2idx, item2idx, idx2item):
        self.matrix    = matrix
        self.user2idx  = user2idx
        self.item2idx  = item2idx
        self.idx2item  = idx2item
        self.user_seen = (
            train_df.groupby("user_id")["product_id"]
            .apply(set).to_dict()
        )
        # Normalised item-user matrix for cosine similarity
        item_matrix        = matrix.T.tocsr()                # (items × users)
        self.norm_item_mat = l2_normalize_rows(item_matrix)  # (items × users)
        return self

    def recommend(self, user_id: str, k: int = 10) -> list:
        if user_id not in self.user2idx:
            return []

        u_idx      = self.user2idx[user_id]
        user_row   = self.matrix[u_idx]                      # (1 × items)
        rated_idx  = user_row.nonzero()[1]

        if len(rated_idx) == 0:
            return []

        user_scores = np.array(user_row[0, rated_idx]).flatten()  # (r,)

        # Item-item cosine similarities for rated items only
        rated_norm  = self.norm_item_mat[rated_idx]          # (r × users)
        # Sim matrix: (all_items × r)
        sim_mat     = self.norm_item_mat @ rated_norm.T       # (items × r)
        agg_scores  = np.array(sim_mat @ user_scores[:, np.newaxis]).flatten()

        seen = self.user_seen.get(user_id, set())
        for p in seen:
            if p in self.item2idx:
                agg_scores[self.item2idx[p]] = -np.inf

        top_k = np.argsort(agg_scores)[-k:][::-1]
        return [self.idx2item[i] for i in top_k if agg_scores[i] > -np.inf]


print("✅ All recommender classes defined.")

## Instantiate and fit all models

In [ ]:
K = 10  # recommendation list length (changed to other values)

shared_fit_args = dict(
    train_df = train_df,
    matrix   = matrix,
    user2idx = user2idx,
    item2idx = item2idx,
    idx2item = idx2item,
)

MODELS = {
    "GlobalPopularity":      GlobalPopularityRecommender(),
    "CategoryPopularity":    CategoryPopularityRecommender(),
    "UserCF  (cosine, N=50)": UserBasedCF(n_neighbors=50),
    "ItemCF  (cosine)":       ItemBasedCF(),
}

for name, model in MODELS.items():
    print(f"⏳  Training  {name:<30}", end=" … ")
    model.fit(**shared_fit_args)
    print("done ✅")

## Multi-K Vectorized Evaluation

---



In [ ]:
# ============================================================
# Evaluates all models at K = 10, 15, 30, 50
# ============================================================

import time
import numpy as np
import pandas as pd
from tabulate import tabulate

K_VALUES = [5, 10, 15, 30, 50]

# ── Restrict to users seen in training ────────────────────
test_known  = test_df[test_df["user_id"].isin(user2idx)].copy()
test_users  = test_known["user_id"].values
test_truths = test_known["product_id"].values
N_TEST      = len(test_users)

print(f"Test users : {N_TEST:,}  |  K values = {K_VALUES}\n")


# ──────────────────────────────────────────────────────────
# METRIC HELPER
# Accepts a single effective_ranks array and computes
# Hit, MRR, NDCG at ALL requested K values in one pass.
# ──────────────────────────────────────────────────────────
def vectorized_metrics_multi_k(effective_ranks_0idx: np.ndarray,
                                k_values: list) -> dict:
    """
    Parameters
    ----------
    effective_ranks_0idx : (n,) int array
        0-indexed rank of truth item after excluding seen items.
        Values >= max(k_values) signal "not found".
    k_values : list[int]

    Returns flat dict:
        { "N_eval": n, "Hit@10": ..., "MRR@10": ..., "NDCG@10": ...,
                       "Hit@15": ..., ... }
    """
    n      = len(effective_ranks_0idx)
    result = {"N_eval": n}

    ranks_1idx = effective_ranks_0idx + 1     # 1-indexed, shape (n,)

    for k in k_values:
        in_topk = effective_ranks_0idx < k    # bool (n,)

        hits  = in_topk.astype(np.float32)
        mrrs  = np.where(in_topk, 1.0 / ranks_1idx,              0.0)
        ndcgs = np.where(in_topk, 1.0 / np.log2(ranks_1idx + 1), 0.0)

        result[f"Hit@{k}"]  = float(np.mean(hits))
        result[f"MRR@{k}"]  = float(np.mean(mrrs))
        result[f"NDCG@{k}"] = float(np.mean(ndcgs))

    return result


# ──────────────────────────────────────────────────────────
# EVALUATOR 1 — GlobalPopularity
# ──────────────────────────────────────────────────────────
def eval_global_popularity(model, test_users, test_truths, k_values):
    t0 = time.time()

    item_to_grank = {item: r for r, item in enumerate(model.popular)}
    n_items       = len(model.popular)

    truth_granks = np.array(
        [item_to_grank.get(t, n_items) for t in test_truths],
        dtype=np.int32
    )

    seen_above = np.array([
        sum(1 for s in model.user_seen.get(u, set())
            if item_to_grank.get(s, n_items) < truth_granks[i])
        for i, u in enumerate(test_users)
    ], dtype=np.int32)

    effective_ranks = truth_granks - seen_above

    print(f"    done in {time.time() - t0:.1f}s")
    return vectorized_metrics_multi_k(effective_ranks, k_values)


# ──────────────────────────────────────────────────────────
# EVALUATOR 2 — CategoryPopularity
# ──────────────────────────────────────────────────────────
def eval_category_popularity(model, test_users, test_truths, k_values):
    t0 = time.time()

    n_items      = sum(len(v) for v in model.cat_items.values())
    cat_item_rank = {
        cat: {item: r for r, item in enumerate(items)}
        for cat, items in model.cat_items.items()
    }

    # Pre-build global rank lookup dict (avoid O(n) next() scan per user)
    global_rank_lookup = {p: r for r, p in enumerate(model.global_popular)}

    effective_ranks = np.full(len(test_users), n_items, dtype=np.int32)

    for i, (u, truth) in enumerate(zip(test_users, test_truths)):
        cat    = model.user_cat.get(u, "")
        cranks = cat_item_rank.get(cat, {})
        seen   = model.user_seen.get(u, set())

        if truth not in cranks:
            grank = global_rank_lookup.get(truth, n_items)
            effective_ranks[i] = len(cranks) + grank
            continue

        t_rank         = cranks[truth]
        seen_above_cat = sum(1 for s in seen if cranks.get(s, n_items) < t_rank)
        effective_ranks[i] = t_rank - seen_above_cat

    print(f"    done in {time.time() - t0:.1f}s")
    return vectorized_metrics_multi_k(effective_ranks, k_values)


# ──────────────────────────────────────────────────────────
# EVALUATOR 3 — CF Models (UserCF / ItemCF)
# Computes effective_ranks once → reused for all K values
# ──────────────────────────────────────────────────────────
def eval_cf_model(model, test_users, test_truths,
                  k_values, batch_size=200, max_users=None):
    t0 = time.time()

    if max_users and max_users < len(test_users):
        rng = np.random.default_rng(seed=42)
        idx = rng.choice(len(test_users), max_users, replace=False)
        test_users  = test_users[idx]
        test_truths = test_truths[idx]
        print(f"    ⚠️  Subsampled to {max_users:,} users for speed")

    n       = len(test_users)
    n_items = matrix.shape[1]
    MISS    = n_items    # sentinel: truth item ranked beyond all items

    effective_ranks = np.full(n, MISS, dtype=np.int32)

    truth_item_idx = np.array(
        [item2idx.get(t, -1) for t in test_truths], dtype=np.int32
    )

    report_step = max(1, n // 20)

    for batch_start in range(0, n, batch_size):
        batch_end = min(batch_start + batch_size, n)

        b_users     = test_users[batch_start:batch_end]
        b_truth_idx = truth_item_idx[batch_start:batch_end]

        u_idx_batch = np.array(
            [model.user2idx.get(u, -1) for u in b_users], dtype=np.int32
        )

        valid_mask      = u_idx_batch >= 0
        valid_positions = np.where(valid_mask)[0]

        if len(valid_positions) == 0:
            continue

        valid_u_idx = u_idx_batch[valid_positions]

        # ── Score matrix for this batch ───────────────────
        if hasattr(model, 'norm_matrix'):
            # UserCF
            batch_norm = model.norm_matrix[valid_u_idx]        # (v × items)
            sim_mat    = batch_norm @ model.norm_matrix.T      # (v × users)
            scores_2d  = np.asarray((sim_mat @ model.matrix).todense())

        elif hasattr(model, 'norm_item_mat'):
            # ItemCF
            batch_hist = model.matrix[valid_u_idx]             # (v × items)
            temp       = batch_hist @ model.norm_item_mat      # (v × users)
            scores_2d  = np.asarray((temp @ model.norm_item_mat.T).todense())

        else:
            raise ValueError("Unknown CF model type")

        if scores_2d.ndim == 1:
            scores_2d = scores_2d.reshape(1, -1)

        assert scores_2d.shape == (len(valid_positions), n_items), \
            f"Shape mismatch: {scores_2d.shape} vs ({len(valid_positions)}, {n_items})"

        # ── Rank extraction ───────────────────────────────
        for j, batch_pos in enumerate(valid_positions):
            global_pos = batch_start + batch_pos

            t_idx = b_truth_idx[batch_pos]
            if t_idx < 0:
                continue

            scores = scores_2d[j].copy()

            u = b_users[batch_pos]
            for p in model.user_seen.get(u, set()):
                ci = model.item2idx.get(p, -1)
                if ci >= 0:
                    scores[ci] = -np.inf

            truth_score = scores[t_idx]
            if not np.isfinite(truth_score):
                continue

            effective_ranks[global_pos] = int((scores > truth_score).sum())

        # ── Progress ──────────────────────────────────────
        done = batch_end
        if done % report_step < batch_size or done == n:
            elapsed = time.time() - t0
            rate    = done / max(elapsed, 1e-6)
            eta     = (n - done) / rate
            print(f"    {done/n*100:5.1f}%  |  {done:,}/{n:,}  |  "
                  f"{rate:.0f} users/s  |  ETA {eta:.0f}s", end="\r")

    print(f"    100.0%  |  {n:,}/{n:,}  |  done in {time.time()-t0:.1f}s          ")
    return vectorized_metrics_multi_k(effective_ranks, k_values)


# ──────────────────────────────────────────────────────────
# DISPATCH
# ──────────────────────────────────────────────────────────
def evaluate(name, model, test_users, test_truths, k_values):
    print(f"⏳  {name}")
    if isinstance(model, GlobalPopularityRecommender):
        return eval_global_popularity(model, test_users, test_truths, k_values)
    elif isinstance(model, CategoryPopularityRecommender):
        return eval_category_popularity(model, test_users, test_truths, k_values)
    else:
        return eval_cf_model(model, test_users, test_truths, k_values,
                             batch_size=200, max_users=None)


# ──────────────────────────────────────────────────────────
# RUN — skip already-cached results
# ──────────────────────────────────────────────────────────
results = {}
for name, model in MODELS.items():
    if name in results:
        print(f"⏭️   {name} — cached, skipping")
        continue
    results[name] = evaluate(name, model, test_users, test_truths, K_VALUES)

    # Inline summary across all K
    m = results[name]
    summary = "  ".join(
        f"Hit@{k}={m[f'Hit@{k}']:.4f} NDCG@{k}={m[f'NDCG@{k}']:.4f}"
        for k in K_VALUES
    )
    print(f"    {summary}\n")


# ──────────────────────────────────────────────────────────
# OUTPUT — one ranked table per K value
# ──────────────────────────────────────────────────────────
print("\n" + "█" * 72)
print(f"{'█  FULL EVALUATION RESULTS  █':^72}")
print("█" * 72)

for k in K_VALUES:
    rows = sorted(
        [
            [name,
             m["N_eval"],
             round(m[f"Hit@{k}"],  4),
             round(m[f"MRR@{k}"],  4),
             round(m[f"NDCG@{k}"], 4)]
            for name, m in results.items()
        ],
        key=lambda x: x[4],   # sort by NDCG
        reverse=True
    )

    print(f"\n{'─' * 72}")
    print(f"  K = {k}")
    print(f"{'─' * 72}")
    print(tabulate(
        rows,
        headers=["Model", "N Eval", f"Hit@{k}", f"MRR@{k}", f"NDCG@{k}"],
        tablefmt="fancy_grid",
        floatfmt=".4f",
    ))
    print(f"  🥇  Best @ K={k} : {rows[0][0]}  "
          f"(NDCG={rows[0][4]:.4f}  Hit={rows[0][2]:.4f}  MRR={rows[0][3]:.4f})")


# ──────────────────────────────────────────────────────────
# BONUS — Cross-K summary heatmap-style table
# Shows NDCG for every (model × K) combination in one view
# ──────────────────────────────────────────────────────────
print(f"\n{'─' * 72}")
print("  NDCG SUMMARY — all models × all K")
print(f"{'─' * 72}")

summary_rows = [
    [name] + [round(results[name][f"NDCG@{k}"], 4) for k in K_VALUES]
    for name in results
]
# Sort by average NDCG across all K values
summary_rows.sort(
    key=lambda r: sum(r[1:]) / len(K_VALUES),
    reverse=True
)
print(tabulate(
    summary_rows,
    headers=["Model"] + [f"NDCG@{k}" for k in K_VALUES],
    tablefmt="fancy_grid",
    floatfmt=".4f",
))

# Overall winner = highest mean NDCG
best_name = summary_rows[0][0]
best_ndcg = summary_rows[0][1:]
print(f"\n🏆  Overall Best : {best_name}")
print(f"    NDCG values  : " +
      "  ".join(f"@{k}={v:.4f}" for k, v in zip(K_VALUES, best_ndcg)))


# Save to CSV
pd.DataFrame([
    {"Model": name,
     **{f"{metric}@{k}": results[name][f"{metric}@{k}"]
        for metric in ["Hit", "MRR", "NDCG"]
        for k in K_VALUES}}
    for name in results
]).to_csv("basic_results_(15,5).csv", index=False)
print("\n✅ Results saved to basic_results_(5,3).csv")